# 05 · dorothea: Kr energy resolution

**Purpose.** Reproduce the 2025 Kr results with the same cuts and fits, for fibers and for PMTs.

**Inputs.** `.dorothea.h5` kDSTs (`/DST/Events`): the merged 2025 files or the new per-file outputs.

**Outputs.** Resolution numbers (printed) and plots.

**What is checked.** R < 30 mm cut, lifetime fit (ln of the mean S2e in 10 equal-count drift-time groups), lifetime correction, one-Gaussian fit. 2025 fibers result: 8.37% FWHM raw, 3.84% after correction, τ = 12.44 ± 0.45 ms (input 12 ms), 5165 events.

**Next step.** `06_qbb`. Run with `./submit.sh dorothea <fibers|pmts> <sample>`.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

In [ ]:
EL_GAIN = paths.EL_GAIN   # gain of the new runs to read (folder <det>_g<EL_GAIN>)
def load_dst(sample, det, use_new=False):
    fs = paths.new_files(sample, det, "dorothea", EL_GAIN) if use_new else paths.old_files(sample, det, "dorothea")
    return pd.concat([pd.read_hdf(f, "/DST/Events") for f in fs], ignore_index=True), fs

RCUT = 30          # mm
V_DRIFT = 1.0      # mm/us (detsim drift_velocity)

def kr_resolution(dst, label, weighted=False, bins_raw=np.linspace(6.5e3, 1.05e4, 50), bins_cor=np.linspace(6.5e3, 1.07e4, 100)):
    d  = dst[dst.R < RCUT]
    dt = d.Z.values / V_DRIFT
    raw = fits.fit_peak(d.S2e.values, bins_raw, weighted)
    lt  = fits.fit_lifetime(dt, d.S2e.values, n=10, weighted=weighted)
    cor = fits.fit_peak(d.S2e.values * np.exp(dt / lt["tau_us"]), bins_cor, weighted)
    print(f"{label} [{'weighted fits' if weighted else '2025 method'}]: {len(d)} events with R < {RCUT} mm")
    print(f"   raw       : mu {raw['mu']:.0f} pes, FWHM {raw['fwhm_pct']:.2f} ± {raw['fwhm_pct_err']:.2f} %")
    print(f"   lifetime  : {lt['tau_us']/1e3:.2f} ± {lt['tau_err_us']/1e3:.2f} ms (input 12 ms)")
    print(f"   corrected : mu {cor['mu']:.0f} pes, FWHM {cor['fwhm_pct']:.2f} ± {cor['fwhm_pct_err']:.2f} %")
    return dict(raw=raw, lt=lt, cor=cor, d=d)

## 2025 fibers result (50Kev sample)

In [ ]:
dst50, fs = load_dst("20250626_Next100_50Kev_ELoff_Kr", "fibers"); print(fs)
r50  = kr_resolution(dst50, "fibers, 50Kev")                  # same method as the 2025 notebook
r50w = kr_resolution(dst50, "fibers, 50Kev", weighted=True)   # Poisson-weighted fits, for comparison

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].errorbar(r50["lt"]["x"], r50["lt"]["y"], r50["lt"]["ey"], fmt="o"); t = np.linspace(0, 1200, 50)
ax[0].plot(t, r50["lt"]["e0"] * np.exp(-t / r50["lt"]["tau_us"])); ax[0].set_xlabel("drift time (µs)"); ax[0].set_ylabel("mean S2e (pes)")
for a, k, title in [(ax[1], "raw", "raw"), (ax[2], "cor", "lifetime corrected")]:
    r = r50[k]; a.step(r["x"], r["y"], where="mid"); a.plot(r["x"], fits.gauss(r["x"], *r["p"]))
    a.set_title(f"{title}: {r['fwhm_pct']:.2f}% FWHM"); a.set_xlabel("S2e (pes)")
plt.tight_layout()

## Fibers vs PMTs on the same nexus events (1Kev sample)
Same cuts and fits. Expected from photostatistics + Fano (Claude's estimate with el_gain 365, not a result): about 3.3% fibers, 3.9% PMTs.

With R < 30 mm the 1Kev sample has only ~150 events per detector, too few for a precise comparison. Better: run the PMTs on the 50Kev sample (`./submit.sh detsim pmts 20250626_Next100_50Kev_ELoff_Kr`, then hypathia and dorothea).

In [ ]:
res = {}
for det in ["fibers", "pmts"]:
    dst, fs = load_dst("20250624_Next100_1Kev_ELoff_Kr", det); print(fs)
    res[det] = kr_resolution(dst, f"{det}, 1Kev")
print(f"S2 ratio fibers/PMTs (corrected peak): {res['fibers']['cor']['mu']/res['pmts']['cor']['mu']:.3f}")